# ML4SCI EXXA: Sequential Test (Exoplanet Light Curves)
**Applicant:** Arya Wadhwa (@aryawadhwa)

This script completes the Sequential Test requirement (for EXXA3 / EXXA4 / EXXA5). It generates synthetic stellar light curves, simulates planetary transits (periodic dimming), adds observational noise, and trains a 1D Convolutional Neural Network (CNN) to classify whether a given light curve contains an exoplanet transit. Finally, it evaluates the model using ROC curves and AUC metrics.

## 1. Import Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 2. Simulate Exoplanet Transit Data
We generate synthetic light curves. A star without a planet is just Gaussian noise around a normalized flux of 1.0. A star with an exoplanet includes periodic U-shaped dips (simulating the physical obscuration of the star's light as the planet transits).

In [ ]:
def generate_light_curve(num_points=500, has_planet=False):
    time = np.linspace(0, 20, num_points)  # 20 days of observation
    flux = np.ones(num_points)
    
    if has_planet:
        # Randomize transit parameters to prevent overfitting
        period = np.random.uniform(3, 8)       # Time between transits (days)
        duration = np.random.uniform(0.2, 0.8) # How long the transit lasts
        depth = np.random.uniform(0.01, 0.05)  # How much star light is blocked (1% to 5%)
        phase = np.random.uniform(0, period)   # When the first transit happens
        
        # Create the periodic dipping
        for i, t in enumerate(time):
            if (t - phase) % period < duration:
                flux[i] -= depth  # Block the light
                
    # Add realistic telescopic observation noise (Gaussian)
    noise_level = np.random.uniform(0.002, 0.008)  # High noise simulating real instruments
    noise = np.random.normal(0, noise_level, num_points)
    return flux + noise

# Create Dataset: 2000 light curves (1000 with planets, 1000 empty stars)
NUM_SAMPLES = 2000
SEQ_LEN = 500

X, y = [], []
for i in range(NUM_SAMPLES):
    has_planet = bool(i % 2) # Alternate exactly 50/50
    X.append(generate_light_curve(SEQ_LEN, has_planet))
    y.append(1 if has_planet else 0)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"Dataset Shape: X={X.shape}, y={y.shape}")

## 3. Visualize the Synthetic Data

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)

# Sample Null (No Planet)
idx_no_planet = np.where(y == 0)[0][0]
axes[0].plot(X[idx_no_planet], color='gray', alpha=0.8)
axes[0].set_title("Synthetic Light Curve: No Planet (Noise Only)")
axes[0].set_ylabel("Normalized Flux")

# Sample Transit (Has Planet)
idx_planet = np.where(y == 1)[0][0]
axes[1].plot(X[idx_planet], color='blue', alpha=0.8)
axes[1].set_title("Synthetic Light Curve: Exoplanet Transit Detected (Periodic Dips)")
axes[1].set_ylabel("Normalized Flux")
axes[1].set_xlabel("Time (Observation Frames)")

plt.tight_layout()
plt.show()

## 4. PyTorch DataLoader

In [ ]:
class LightCurveDataset(Dataset):
    def __init__(self, sequences, labels):
        # Reshape to (Batch, Channels, Length) for 1D Convolutions
        self.X = torch.tensor(sequences).unsqueeze(1)
        self.y = torch.tensor(labels).unsqueeze(1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_loader = DataLoader(LightCurveDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(LightCurveDataset(X_test, y_test), batch_size=32, shuffle=False)
print("Dataloaders initialized.")

## 5. 1D Convolutional Neural Network (Classifier)

In [ ]:
class TransitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),
            
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),
            
            nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid() # Output a probability [0, 1] of a planet existing
        )
        
    def forward(self, x):
        features = self.conv_blocks(x)
        features = features.squeeze(-1) # Flatten
        return self.classifier(features)

model = TransitCNN().to(DEVICE)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

print(f"Total Parameters: {sum(p.numel() for p in model.parameters())}")

## 6. Training Loop

In [ ]:
EPOCHS = 20
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    train_losses.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}	Loss: {train_losses[-1]:.4f}")

plt.figure(figsize=(6, 3))
plt.plot(train_losses, label="Binary Cross Entropy Loss")
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

## 7. Testing & ROC Curve (Withheld Data)

In [ ]:
model.eval()
all_preds = []
all_truths = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(DEVICE)
        predictions = model(batch_x)
        
        all_preds.extend(predictions.cpu().numpy())
        all_truths.extend(batch_y.numpy())

all_preds = np.array(all_preds).flatten()
all_truths = np.array(all_truths).flatten()

# Calculate False Positive Rate, True Positive Rate, and Area Under the Curve (AUC)
fpr, tpr, thresholds = roc_curve(all_truths, all_preds)
roc_auc = auc(fpr, tpr)

# Calculate basic accuracy at 0.5 threshold
binary_preds = (all_preds > 0.5).astype(int)
accuracy = (binary_preds == all_truths).mean() * 100

print(f"Test Set Accuracy: {accuracy:.2f}%")
print(f"ROC AUC Score: {roc_auc:.4f}")

fig, ax = plt.subplots(figsize=(6, 6))
display = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc, estimator_name="TransitCNN")
display.plot(ax=ax)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guess')
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.show()

---
### Inference Pipeline Verification
Demonstrating running inference on a single piece of withheld data.

In [ ]:
sample_idx = 42
sample_x = torch.tensor(X_test[sample_idx]).unsqueeze(0).unsqueeze(1).to(DEVICE)
ground_truth = y_test[sample_idx]

model.eval()
with torch.no_grad():
    prob = model(sample_x).item()

print(f"Inference on withheld sample {sample_idx}:")
print(f"Ground Truth: {'Planet Present' if ground_truth == 1 else 'Empty Star'}")
print(f"Model Prediction Confidence: {prob:.4f}")
if prob > 0.5:
    print("Model Classification: planet detected.")
else:
    print("Model Classification: no planet.")